In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs
import geopandas as gpd
from shapely.ops import unary_union

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib.patches as mpatches
import gsw

In [ ]:
def Calculate_TEOS10_vars(ds):

    if 'sal' not in ds:     # this skips D/2017/ctd which is not a dataset 
        return ds
    
    ds = ds.copy()  # important: DatasetView → writable Dataset

    ds['abs_sal'] = (
        'time',
        gsw.SA_from_SP(
            SP=ds['sal'].values,
            p=ds['z'].values,
            lon=ds['lon'].values,
            lat=ds['lat'].values,
        )
    )

    ds['theta'] = (
        'time',
        gsw.pt_from_t(
            SA=ds['abs_sal'].values,
            t=ds['temp'].values,
            p=ds['z'].values,
            p_ref=0,
        )
    )

    ds['CT'] = (
        'time',
        gsw.CT_from_pt(
            SA=ds['abs_sal'].values,
            pt=ds['theta'].values,
        )
    )

    ds['sigma0'] = (
        'time',
        gsw.sigma0(
            SA=ds['abs_sal'].values,
            CT=ds['CT'].values,
        )
    )

    return ds

import numpy as np

def calculate_direction(east,north):
    """
    This function calculates direction given east and north velocity components 

    ------
    Parameters 
    ------
    Inputs:
        - east: east velocity component; array 
        - north: north velocity component; array

    Outputs:
        - direction: calculated direction [degrees]; array 
    """
    
    direction = np.mod(np.arctan2(east,north)*180/np.pi,360)

    return direction 
    

def calculate_true_direction(direction,magnetic_declination):
    """
    Function that calculates true direction given a direction array in degrees and magnetic declination
    in degrees 

    -----
    Parameters
    -----

    Inputs: 
        - direction: array; direction in degrees 
        - magnetic_declination: float; magnetic declination angle in degrees 

    Outputs: 
        - true_direction: array; true direction in degrees (i.e. direction accounted for magnetic 
        declination) 
    
    """

    true_direction = np.mod(direction+magnetic_declination,360)

    return true_direction 
    
def rotate_velocity(u,v,angle):
    """Function that rotates velocity component vectors using rotation equations
       
       Inputs: v: u velocity, v velocity, angle: angle between vectors [radians]
       Outputs: rotated v velocity vectors 
    """
    u_prime = u*np.cos(angle) - v*np.sin(angle)
    v_prime = u*np.sin(angle) + v*np.cos(angle)
    
    return u_prime,v_prime



In [ ]:
bath = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_Research/data/gebco_2024_n-66.2651_s-87.7985_w159.418_e213.2403.nc')

In [ ]:
# Mooring location (constant lat/lon)
tnbd_lat = -75.0112
tnbd_lon = 165.5562

ditd_lat = -75.2751
ditd_lon = 164.0625

D_lat = -75.133
D_lon = 164.542

ditn_lat = -75.36083
ditn_lon = 164.7463

p2_lat = -71.4601
p2_lon = 172.3024

p3_lat = -71.9181
p3_lon = 172.9265

ldeo_lat = -74.97
ldeo_lon = 163.96

dits_lat = -75.4886
dits_lon = 163.2042

In [ ]:
gdf = gpd.read_file("/Users/mackenzie/Documents/Research/PhD/TNB_Research/data/add_coastline_high_res_polygon_v7.2/add_coastline_high_res_polygon_v7.2.shp")

In [ ]:
# make sure it is in lon/lat
if gdf.crs is not None:
    gdf = gdf.to_crs("EPSG:4326")

ice = gdf[gdf["surface"].isin(["ice shelf", "ice tongue"])]
land = gdf[gdf["surface"].isin(["land"])]

In [ ]:
ice

In [ ]:
from shapely.geometry import box

region = box(162, -75.75, 170, -74.25)
ice_clip = ice.clip(region)
land_clip = land.clip(region)

In [ ]:
# starting from scratch... this is so fucking annoying 

# step 1: set lat/lon extent 
# --- Define your region ---
lon_min, lon_max = 150, 180
lat_min, lat_max = -80, -70

# step 2: 
proj = ccrs.SouthPolarStereo(central_longitude=165)

fig, ax = plt.subplots(subplot_kw={"projection": proj})

ax.set_extent([162, 170, -75.75, -74.25], crs=ccrs.PlateCarree())

gl = ax.gridlines(
    crs=ccrs.PlateCarree(),
    draw_labels=True,
    linewidth=0.5,
    color='gray',
    alpha=0.5,
    linestyle='--'
)

gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {'size': 10, 'weight': 'bold'}
gl.ylabel_style = {'size': 10, 'weight': 'bold'}

# set latitude tick marks 
gl.ylocator = plt.FixedLocator([-75.6,-75, -74.4])
depth = -bath.sel(lat=slice(lat_min,lat_max),lon=slice(lon_min,lon_max))['elevation']

levels = [1000]

contours = ax.contour(
    depth['lon'],
    depth['lat'],
    depth,
    levels=levels,
    colors='black',
    linewidths=0.7,
    transform=ccrs.PlateCarree(),
    zorder=2
)
ax.clabel(contours, fmt='%d m', fontsize=8, inline=True, inline_spacing=1)

ax.add_geometries(
    ice_clip.geometry,
    crs=ccrs.PlateCarree(),
    facecolor='lightgrey',
    edgecolor='k',
    linewidth=0.3,
    zorder=3
)

ax.add_geometries(
    land_clip.geometry,
    crs=ccrs.PlateCarree(),
    facecolor='grey',
    edgecolor='k',
    linewidth=0.3,
    zorder=3
)


In [ ]:
# make into a function 
def TNB_study_site():
    """
    """
    # step 2: 
    proj = ccrs.SouthPolarStereo(central_longitude=165)

    fig, ax = plt.subplots(subplot_kw={"projection": proj})

    ax.set_extent([162, 170, -75.75, -74.25], crs=ccrs.PlateCarree())

    gl = ax.gridlines(
        crs=ccrs.PlateCarree(),
        draw_labels=True,
        linewidth=0.5,
        color='gray',
        alpha=0.5,
        linestyle='--'
    )

    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {'size': 10, 'weight': 'bold'}
    gl.ylabel_style = {'size': 10, 'weight': 'bold'}
    # set latitude tick marks 
    gl.ylocator = plt.FixedLocator([-75.6,-75, -74.4])

    depth = -bath.sel(lat=slice(lat_min,lat_max),lon=slice(lon_min,lon_max))['elevation']

    levels = [1000]

    contours = ax.contour(
        depth['lon'],
        depth['lat'],
        depth,
        levels=levels,
        colors='black',
        linewidths=0.7,
        transform=ccrs.PlateCarree(),
        zorder=2
    )
    ax.clabel(contours, fmt='%d m', fontsize=8, inline=True, inline_spacing=1)

    ax.add_geometries(
        ice_clip.geometry,
        crs=ccrs.PlateCarree(),
        facecolor='lightgrey',
        edgecolor='k',
        linewidth=0.3,
        zorder=3
    )

    ax.add_geometries(
        land_clip.geometry,
        crs=ccrs.PlateCarree(),
        facecolor='grey',
        edgecolor='k',
        linewidth=0.3,
        zorder=3
    )
    return fig, ax


In [ ]:
fig, ax = TNB_study_site()

In [ ]:
# load in velocity data for D, TNBD, and DITD 


In [ ]:
# load in data from MAC 
# DITD
DITD_ctd_top = xr.open_dataset('DITD_2018_ctd_top.nc')
DITD_ctd_bot = xr.open_dataset('DITD_2018_ctd_bot.nc')
DITD_vel_top = xr.open_dataset('vel_1100.nc')
DITD_vel_bot = xr.open_dataset('vel_1200.nc')

# TNBD 
TNBD_ctd_bot = xr.open_dataset('tnbd_2018_ctd_bot_1093.nc')
TNBD_ctd_top = xr.open_dataset('tnbd_2018_ctd_top_999.nc')
TNBD_vel_bot = xr.open_dataset('tnbd_2018_vel_bot_1093.nc')
TNBD_vel_top = xr.open_dataset('tnbd_2018_vel_top_999.nc')



In [ ]:
# load in moorings datatree for mooring D data 
moorings = xr.open_datatree('/Users/mackenzie/Documents/Research/PhD/TNB_tools/preprocess/TNB_Moorings_DataTree.nc')

In [ ]:
D_vel_bot = moorings['D/2017/vel/bot'].to_dataset().sel(time=slice('2018-03-01','2019-01-01'))
D_vel_mid = moorings['D/2017/vel/mid'].to_dataset().sel(time=slice('2018-03-01','2019-01-01'))
D_ctd_mid = moorings['D/2017/ctd/mid'].to_dataset().sel(time=slice('2018-03-01','2019-01-01'))

In [ ]:
D_ctd_mid = Calculate_TEOS10_vars(D_ctd_mid)

In [ ]:
ldeo = moorings['LDEO/2017/ctd/bot'].to_dataset()

In [ ]:
# load in relevant data - TNBD and DITD ON DELL 
# DITD_ctd_top = xr.open_dataset('/Users/mblan/Documents/TNB_Research/data/DITD_2018_2019/DITD_2018_ctd_top.nc')
# DITD_ctd_bot = xr.open_dataset('/Users/mblan/Documents/TNB_Research/data/DITD_2018_2019/DITD_2018_ctd_bot.nc')
# DITD velocity data (NOTE TIME IS NOT CORRECTED IN THIS SAVED DATA  - NEED TO CORRECT AND RE-SAVE)
# DITD_vel_top = xr.open_dataset('/Users/mblan/Documents/TNB_Research/data/DITD_2018_2019/vel_1100.nc')
# DITD_vel_bot = xr.open_dataset('/Users/mblan/Documents/TNB_Research/data/DITD_2018_2019/vel_1200.nc')

In [ ]:
import pandas as pd

In [ ]:
# fix DITD time - should save this change permenantly later
DITD_vel_top['time'] = DITD_vel_top.time - pd.Timedelta(12,'h')

In [ ]:
# rename variables in DITD velocity dataset 
DITD_vel_top = DITD_vel_top.rename({
    'east':'u',
    'north':'v',
    'up':'w',
    'pressure':'pres',
    'direction':'dir'
}).assign_coords({
    'z':1121,
    'lat':-75.28,
    'lon':164.1
})
# rename variables in DITD velocity dataset 
DITD_vel_bot = DITD_vel_bot.rename({
    'east':'u',
    'north':'v',
    'up':'w',
    'pressure':'pres',
    'direction':'dir'
}).assign_coords({
    'z':1218,
    'lat':-75.28,
    'lon':164.1
})

In [ ]:
# convert velocity from m/s to cm/s 
DITD_vel_top['u'] = DITD_vel_top.u * 100
DITD_vel_top['v'] = DITD_vel_top.v * 100
DITD_vel_top['speed'] = DITD_vel_top.speed * 100

DITD_vel_bot['v'] = DITD_vel_bot.v * 100
DITD_vel_bot['u'] = DITD_vel_bot.u * 100
DITD_vel_bot['speed'] = DITD_vel_bot.speed * 100


In [ ]:
# crop DITD vel bot to remove bad time at end of timeseries
DITD_vel_bot = DITD_vel_bot.sel(time=slice(None,'2018-06-21T20:00'))

In [ ]:
# # loadi n TNBD data 
# TNBD_ctd_bot = xr.open_dataset('//thepenguin/penguin2/Data/TNB/mooring/LDEO/TNBD_2018/working/tnbd_2018_ctd_bot_1093.nc')
# TNBD_ctd_top = xr.open_dataset('//thepenguin/penguin2/Data/TNB/mooring/LDEO/TNBD_2018/working/tnbd_2018_ctd_top_999.nc')
# TNBD_vel_bot = xr.open_dataset('//thepenguin/penguin2/Data/TNB/mooring/LDEO/TNBD_2018/working/tnbd_2018_vel_bot_1093.nc')
# TNBD_vel_top = xr.open_dataset('//thepenguin/penguin2/Data/TNB/mooring/LDEO/TNBD_2018/working/tnbd_2018_vel_top_999.nc')

In [ ]:
# to save some memory 
del moorings

In [ ]:
# get on same time axis / resample 


In [ ]:
########### ALIGN TNBD ################
# data resolutions 
# ctd bottom - 3 min
# ctd top - 1 min
# vel bot - 15 min
# vel top - 30 min
# I will resample the other datasets to 30 min to do analysis between them 
TNBD_vel_bot = TNBD_vel_bot.resample(time='30min').mean().sel(time=slice('2018-03-14','2019-01-03T23:30'))
TNBD_ctd_bot = TNBD_ctd_bot.resample(time='30min').mean().sel(time=slice('2018-03-14','2019-01-03T23:30'))
TNBD_ctd_top = TNBD_ctd_top.resample(time='30min').mean().sel(time=slice('2018-03-14','2019-01-03T23:30'))

TNBD_vel_top = TNBD_vel_top.sel(time=slice('2018-03-14',None))

In [ ]:
########### ALIGN DITD ############## 

# resample data to same dt 
DITD_ctd_bot = DITD_ctd_bot.resample(time='30min').mean()
DITD_ctd_top = DITD_ctd_top.resample(time='30min').mean()
DITD_vel_bot = DITD_vel_bot.resample(time='30min').mean()
# align times 
DITD_vel_top = DITD_vel_top.resample(time='30min').mean()
DITD_ctd_bot = DITD_ctd_bot.sel(time=slice('2018-03-11',None))
DITD_ctd_top = DITD_ctd_top.sel(time=DITD_ctd_bot.time)

In [ ]:
DITD_vel_top

In [ ]:
D_vel_bot

In [ ]:
TNBD_subset = TNBD_vel_bot.sel(time=slice('2018'))

In [ ]:
TNBD_subset.time.values

In [ ]:
DITD_subset = DITD_vel_top.sel(time=TNBD_subset.time)

In [ ]:
D_subset = D_vel_mid.sel(time=TNBD_subset.time)

In [ ]:
# D_subset = D_vel_mid.sel(time=slice('2018-09-01','2018-10-01'))

In [ ]:
# TNBD_subset = TNBD_vel_bot.sel(time=slice('2018-09-01','2018-10-01'))
# DITD_subset = DITD_vel_top.sel(time=slice('2018-09-01','2018-10-01'))

In [ ]:
# # save intermediate datasets
# D_subset.to_netcdf('D_subset_september2018.nc')
# TNBD_subset.to_netcdf('TNBD_subset_september2018.nc')
# DITD_subset.to_netcdf('DITD_subset_september2018.nc')

In [ ]:
# open intermediate datasets
D_subset = xr.open_dataset('D_subset_september2018.nc')
TNBD_subset = xr.open_dataset('TNBD_subset_september2018.nc')
DITD_subset = xr.open_dataset('DITD_subset_september2018.nc')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from pathlib import Path

def plot_velocity_frame(t, outpath=None, vmin=None, vmax=None):
    fig, ax = TNB_study_site()

    lon = np.array([
        float(D_subset.lon),
        float(TNBD_subset.lon),
        float(DITD_subset.lon)
    ])

    lat = np.array([
        float(D_subset.lat),
        float(TNBD_subset.lat),
        float(DITD_subset.lat)
    ])

    u = np.array([
        float(D_subset.u.isel(time=t)),
        float(TNBD_subset.u.isel(time=t)),
        float(DITD_subset.u.isel(time=t))
    ])

    v = np.array([
        float(D_subset.v.isel(time=t)),
        float(TNBD_subset.v.isel(time=t)),
        float(DITD_subset.v.isel(time=t))
    ])

    speed = np.sqrt(u**2 + v**2)
    u_dir = u / (speed + 1e-12)
    v_dir = v / (speed + 1e-12)

    import matplotlib as mpl

    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)

    q = ax.quiver(
        lon, lat,
        u_dir, v_dir,
        speed,
        cmap="viridis",
        norm=norm,   # 👈 THIS is the fix
        scale=10,
        transform=ccrs.PlateCarree(),
        zorder=10
    )

    cbar = plt.colorbar(q, ax=ax, shrink=0.7)
    cbar.set_label("Speed (cm/s)")

    timestamp = np.datetime_as_string(D_subset.time.isel(time=t).values, unit="m")
    ax.set_title(timestamp)

    if outpath is not None:
        plt.savefig(outpath, dpi=200, bbox_inches="tight")
        plt.close(fig)
    else:
        plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from pathlib import Path

def plot_velocity_frame(t, outpath=None, vmin=None, vmax=None):
    fig, ax = TNB_study_site()

    lon = np.array([
        float(D_subset.lon),
        float(TNBD_subset.lon),
        float(DITD_subset.lon)
    ])

    lat = np.array([
        float(D_subset.lat),
        float(TNBD_subset.lat),
        float(DITD_subset.lat)
    ])

    u = np.array([
        float(D_subset.u.isel(time=t)),
        float(TNBD_subset.u.isel(time=t)),
        float(DITD_subset.u.isel(time=t))
    ])

    v = np.array([
        float(D_subset.v.isel(time=t)),
        float(TNBD_subset.v.isel(time=t)),
        float(DITD_subset.v.isel(time=t))
    ])

    speed = np.sqrt(u**2 + v**2)
    u_dir = u / (speed + 1e-12)
    v_dir = v / (speed + 1e-12)

    speed_cap = 20
    speed_clip = np.clip(speed, 0, speed_cap)
    min_len = 0.01
    max_len = 1.25
    length_scale = min_len + (max_len - min_len) * np.sqrt(speed_clip / speed_cap)

    u_plot = u_dir * length_scale
    v_plot = v_dir * length_scale

    import matplotlib as mpl

    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)

    q = ax.quiver(
        lon, lat,
        u_plot, v_plot,
        speed,
        cmap="viridis",
        norm=norm,
        scale=8,
        transform=ccrs.PlateCarree(),
        zorder=10
    )

    cbar = plt.colorbar(q, ax=ax, shrink=0.7)
    cbar.set_label("Speed (cm/s)")

    timestamp = np.datetime_as_string(D_subset.time.isel(time=t).values, unit="m")
    ax.set_title(timestamp)

    if outpath is not None:
        plt.savefig(outpath, dpi=200, bbox_inches="tight")
        plt.close(fig)
    else:
        plt.show()

In [ ]:
all_speeds = np.concatenate([
    np.sqrt(D_subset.u.values**2 + D_subset.v.values**2),
    np.sqrt(TNBD_subset.u.values**2 + TNBD_subset.v.values**2),
    np.sqrt(DITD_subset.u.values**2 + DITD_subset.v.values**2)
])

vmin = 0
# vmax = np.nanpercentile(all_speeds, 99)
vmax = 20 

In [ ]:
for t in range(100,105):
    plot_velocity_frame(t, vmin=vmin, vmax=vmax)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs
from matplotlib.animation import FFMpegWriter

def make_velocity_mp4(
    outfile="velocity_animation.mp4",
    fps=10,
    dpi=150,
    vmin=None,
    vmax=None,
    speed_cap=25,   # speed at which arrow length stops increasing
    min_len=0.15,   # shortest displayed arrow length
    max_len=1.8,    # longest displayed arrow length
    exponent=0.3,   # lower = more separation among weak speeds
    scale=6         # smaller = longer arrows
):
    # -------------------------
    # Precompute static arrays
    # -------------------------
    lon = np.array([
        float(D_subset.lon),
        float(TNBD_subset.lon),
        float(DITD_subset.lon)
    ])

    lat = np.array([
        float(D_subset.lat),
        float(TNBD_subset.lat),
        float(DITD_subset.lat)
    ])

    u_all = np.vstack([
        D_subset.u.values,
        TNBD_subset.u.values,
        DITD_subset.u.values
    ]).T   # shape: (ntime, 3)

    v_all = np.vstack([
        D_subset.v.values,
        TNBD_subset.v.values,
        DITD_subset.v.values
    ]).T   # shape: (ntime, 3)

    time_all = D_subset.time.values
    speed_all = np.sqrt(u_all**2 + v_all**2)

    nt = len(time_all)

    # If color limits are not given, use full dataset range
    if vmin is None:
        vmin = float(np.nanmin(speed_all))
    if vmax is None:
        vmax = float(np.nanmax(speed_all))

    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)

    # -------------------------
    # Helper for arrow scaling
    # -------------------------
    def get_display_vectors(u, v, speed):
        u_dir = u / (speed + 1e-12)
        v_dir = v / (speed + 1e-12)

        # cap speeds only for arrow length
        speed_clip = np.clip(speed, 0, speed_cap)

        # nonlinear scaling to help distinguish weak speeds
        speed_scaled = (speed_clip / speed_cap) ** exponent

        # map to desired display-length range
        length_scale = min_len + (max_len - min_len) * speed_scaled

        u_plot = u_dir * length_scale
        v_plot = v_dir * length_scale
        return u_plot, v_plot

    # -------------------------
    # Build figure once
    # -------------------------
    fig, ax = TNB_study_site()

    # initialize first frame
    u0 = u_all[0]
    v0 = v_all[0]
    s0 = speed_all[0]

    u_plot0, v_plot0 = get_display_vectors(u0, v0, s0)

    q = ax.quiver(
        lon, lat,
        u_plot0, v_plot0,
        s0,                         # true speed controls color
        cmap="viridis",
        norm=norm,
        scale=scale,
        transform=ccrs.PlateCarree(),
        zorder=10
    )

    cbar = plt.colorbar(q, ax=ax, shrink=0.7)
    cbar.set_label("Speed (cm/s)")

    title = ax.set_title(np.datetime_as_string(time_all[0], unit="m"))

    # -------------------------
    # Write directly to mp4
    # -------------------------
    writer = FFMpegWriter(fps=fps)

    with writer.saving(fig, outfile, dpi=dpi):
        for t in range(nt):
            u = u_all[t]
            v = v_all[t]
            speed = speed_all[t]

            u_plot, v_plot = get_display_vectors(u, v, speed)

            # update existing quiver + title
            q.set_UVC(u_plot, v_plot, speed)
            title.set_text(np.datetime_as_string(time_all[t], unit="m"))

            writer.grab_frame()

    plt.close(fig)

In [ ]:
make_velocity_mp4(
    outfile="AWS_mooring_velocity.mp4",
    fps=10,
    dpi=120,
    speed_cap=20,
    min_len=0.01,
    max_len=1.5,
    exponent=0.3,
    scale=8
)

In [ ]:
frames_dir = Path("frames")
frames_dir.mkdir(exist_ok=True)

nt = min(
    D_subset.sizes["time"],
    TNBD_subset.sizes["time"],
    DITD_subset.sizes["time"]
)

for t in range(1474,nt):
    outfile = frames_dir / f"frame_{t:04d}.png"
    plot_velocity_frame(t, outpath=outfile, vmin=vmin, vmax=vmax)

In [ ]:
import imageio.v2 as imageio
from pathlib import Path

frames_dir = Path("frames")

with imageio.get_writer("TNB_velocity.mp4", fps=6) as writer:
    for t in range(nt):
        fname = frames_dir / f"frame_{t:04d}.png"
        image = imageio.imread(fname)
        writer.append_data(image)

In [ ]:
# import imageio.v2 as imageio

# images = []
# for t in range(nt):
#     fname = frames_dir / f"frame_{t:04d}.png"
#     images.append(imageio.imread(fname))

# imageio.mimsave("TNB_velocity.gif", images, duration=0.15)

### Generalized animation code 

In [ ]:
import cmocean

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs
from matplotlib.animation import FFMpegWriter


def make_velocity_mp4(
    moorings,
    outfile="velocity_animation.mp4",
    map_func=None,
    fps=10,
    dpi=150,
    vmin=0,
    vmax=25,
    speed_cap=25,
    min_len=0.05,
    max_len=1,
    exponent=0.3,
    scale=10,
):
    """
    Animate velocity quivers for any number of moorings.

    Parameters
    ----------
    moorings : list of dict, each with keys:
        - 'name' : str
            Label for the mooring (used in legend/title if desired).
        - 'lon'  : float
            Longitude of the mooring.
        - 'lat'  : float
            Latitude of the mooring.
        - 'u'    : array-like, shape (ntime,)
            Eastward velocity component (cm/s).
        - 'v'    : array-like, shape (ntime,)
            Northward velocity component (cm/s).
        - 'time' : array-like, shape (ntime,)
            Time values (numpy datetime64 or anything datetime_as_string accepts).
            All moorings must share the same time axis.

    outfile  : str   — output .mp4 path
    map_func : callable — returns (fig, ax) on a Cartopy GeoAxes;
                          if None a simple PlateCarree map is built automatically.
    fps      : int   — frames per second
    dpi      : int   — render resolution
    vmin/vmax: float — color-scale limits (auto if None)
    speed_cap: float — speed (cm/s) at which arrow length saturates
    min_len  : float — shortest arrow length in quiver units
    max_len  : float — longest arrow length in quiver units
    exponent : float — nonlinear stretch for weak-speed separation (0 < exp ≤ 1)
    scale    : float — quiver scale parameter (smaller → longer arrows)

    Example
    -------
    moorings = [
        {"name": "M1", "lon": -170.1, "lat": -75.2,
         "u": ds1.u.values, "v": ds1.v.values, "time": ds1.time.values},
        {"name": "M2", "lon": -169.8, "lat": -75.5,
         "u": ds2.u.values, "v": ds2.v.values, "time": ds2.time.values},
    ]
    make_velocity_mp4(moorings, outfile="my_animation.mp4", map_func=my_map_func)
    """

    # ------------------------------------------------------------------
    # 1. Pack mooring data into arrays  (n_moorings, ntime) → transposed
    # ------------------------------------------------------------------
    lons = np.array([float(m["lon"]) for m in moorings])
    lats = np.array([float(m["lat"]) for m in moorings])

    u_all = np.vstack([np.asarray(m["u"]) for m in moorings]).T   # (ntime, n_moorings)
    v_all = np.vstack([np.asarray(m["v"]) for m in moorings]).T
    time_all = np.asarray(moorings[0]["time"])
    speed_all = np.sqrt(u_all**2 + v_all**2)

    nt = len(time_all)

    # ------------------------------------------------------------------
    # 2. Color normalization
    # ------------------------------------------------------------------
    if vmin is None:
        vmin = float(np.nanmin(speed_all))
    if vmax is None:
        vmax = float(np.nanmax(speed_all))

    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)

    # ------------------------------------------------------------------
    # 3. Arrow-length helper
    # ------------------------------------------------------------------
    def get_display_vectors(u, v, speed):
        u_dir = u / (speed + 1e-12)
        v_dir = v / (speed + 1e-12)
        speed_clip = np.clip(speed, 0, speed_cap)
        speed_scaled = (speed_clip / speed_cap) ** exponent
        length_scale = min_len + (max_len - min_len) * speed_scaled
        return u_dir * length_scale, v_dir * length_scale

    # ------------------------------------------------------------------
    # 4. Build figure — use supplied map function or a plain fallback
    # ------------------------------------------------------------------
    if map_func is not None:
        fig, ax = map_func()
    else:
        # Auto-build a simple regional map centred on the moorings
        lon_pad = max((lons.max() - lons.min()) * 0.5, 1.0)
        lat_pad = max((lats.max() - lats.min()) * 0.5, 1.0)
        extent = [lons.min() - lon_pad, lons.max() + lon_pad,
                  lats.min() - lat_pad, lats.max() + lat_pad]

        fig = plt.figure(figsize=(8, 6))
        ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
        ax.set_extent(extent, crs=ccrs.PlateCarree())
        try:
            import cartopy.feature as cfeature
            ax.add_feature(cfeature.LAND, zorder=5)
            ax.add_feature(cfeature.COASTLINE, zorder=6)
            ax.gridlines(draw_labels=True)
        except Exception:
            pass

    # ------------------------------------------------------------------
    # 5. Optionally label each mooring (scatter + text)
    # ------------------------------------------------------------------
    for m, lon_m, lat_m in zip(moorings, lons, lats):
        ax.plot(lon_m, lat_m, "k.", markersize=4,
                transform=ccrs.PlateCarree(), zorder=11)
        ax.text(lon_m, lat_m, f"  {m['name']}", fontsize=7,
                transform=ccrs.PlateCarree(), zorder=12)

    # ------------------------------------------------------------------
    # 6. Initial quiver
    # ------------------------------------------------------------------
    u_plot0, v_plot0 = get_display_vectors(u_all[0], v_all[0], speed_all[0])

    q = ax.quiver(
        lons, lats,
        u_plot0, v_plot0,
        speed_all[0],
        cmap='turbo',
        norm=norm,
        scale=scale,
        transform=ccrs.PlateCarree(),
        zorder=10,
    )

    cbar = plt.colorbar(q, ax=ax, shrink=0.7)
    cbar.set_label("Speed (cm/s)")
    title = ax.set_title(np.datetime_as_string(time_all[0], unit="m"))

    # ------------------------------------------------------------------
    # 7. Animation loop
    # ------------------------------------------------------------------
    writer = FFMpegWriter(fps=fps)

    with writer.saving(fig, outfile, dpi=dpi):
        for t in range(nt):
            u_plot, v_plot = get_display_vectors(u_all[t], v_all[t], speed_all[t])
            q.set_UVC(u_plot, v_plot, speed_all[t])
            title.set_text(np.datetime_as_string(time_all[t], unit="m"))
            writer.grab_frame()

    plt.close(fig)

In [ ]:
# open intermediate datasets
# D_subset = xr.open_dataset('D_subset_september2018.nc').sel(time=slice('2018-09-15','2018-09-20'))
# TNBD_subset = xr.open_dataset('TNBD_subset_september2018.nc').sel(time=slice('2018-09-15','2018-09-20'))
# DITD_subset = xr.open_dataset('DITD_subset_september2018.nc').sel(time=slice('2018-09-15','2018-09-20'))

In [ ]:
# TNBD_subset.to_netcdf('TNBD_subset_30min_05262026.nc')

In [ ]:
TNBD_subset 

In [ ]:
# D_subset.to_netcdf('D_subset_30min_05262026.nc')

In [ ]:
# DITD_subset.to_netcdf('DITD_subset_30min_05262026.nc')

In [ ]:
# average datasets to two day averages 
# TNBD_subset = TNBD_subset.resample(time='7D').mean()

In [ ]:
# DITD_subset = DITD_subset.resample(time='7D').mean()

In [ ]:
# D_subset = D_subset.resample(time='7D').mean()

In [ ]:
# D_subset

In [ ]:
DITD_sub = DITD_subset.resample(time='3h').mean()
TNBD_sub = TNBD_subset.resample(time='3h').mean()
D_sub = D_subset.resample(time='3h').mean()

In [ ]:
D_subset

In [ ]:
moorings = [
    {"name": "D",    "lon": float(D_subset.lon),    "lat": float(D_subset.lat),
     "u": D_subset.u.values,    "v": D_subset.v.values,    "time": D_subset.time.values},
    {"name": "TNBD", "lon": float(TNBD_subset.lon), "lat": float(TNBD_subset.lat),
     "u": TNBD_subset.u.values, "v": TNBD_subset.v.values, "time": TNBD_subset.time.values},
    {"name": "DITD", "lon": float(DITD_subset.lon), "lat": float(DITD_subset.lat),
     "u": DITD_subset.u.values, "v": DITD_subset.v.values, "time": DITD_subset.time.values},
]

# moorings = [
#     {"name": "D",    "lon": float(D_sub.lon),    "lat": float(D_sub.lat),
#      "u": D_sub.u.values,    "v": D_sub.v.values,    "time": D_sub.time.values},
#     {"name": "TNBD", "lon": float(TNBD_sub.lon), "lat": float(TNBD_sub.lat),
#      "u": TNBD_sub.u.values, "v": TNBD_sub.v.values, "time": TNBD_sub.time.values},
#     {"name": "DITD", "lon": float(DITD_sub.lon), "lat": float(DITD_sub.lat),
#      "u": DITD_sub.u.values, "v": DITD_sub.v.values, "time": DITD_sub.time.values},
# ]

make_velocity_mp4(moorings, map_func=TNB_study_site, outfile="velocity_animation_30min_average.mp4")

In [ ]:
TNBD_subset.z.values